# SadTalker on Google Colab (T4) — Full Working Notebook

Generates a lip-synced talking-avatar video from **one image + one audio file**, and includes an optional **web UI** (Gradio) so you don't have to edit code for every run.

### Why the setup looks the way it does
Colab's default Python (3.11/3.12) is too new for SadTalker's pinned dependencies, so this notebook builds an isolated **Python 3.10 conda environment** (via `condacolab`) instead of using Colab's system Python. A few one-line patches are baked in below for real errors hit while testing this exact setup:
- `setuptools<81` — newer setuptools removed `pkg_resources`, which an old pinned `librosa` still needs
- `MPLBACKEND=Agg` — stops Colab's notebook-display matplotlib backend from crashing headless script runs
- system `ffmpeg` symlink — conda-forge's `ffmpeg` build had a broken `libx264` link; Colab's built-in `ffmpeg` (via apt) works correctly

### Run order (do not skip steps or run out of order)
| Step | Cell | What it does |
|---|---|---|
| 1 | Cell 1 | Installs condacolab -> runtime auto-restarts (expected) |
| 2 | Cell 2 | Creates Python 3.10 env, installs SadTalker + all dependency patches, downloads model weights |
| 3 | Cell 3 | Upload your avatar image + audio file |
| 4A | Cell 4A | Option A: Command-line inference (fast, one-shot, edit settings in code) |
| 4B | Cell 4B | Option B: Launch Gradio web UI (drag-and-drop browser interface) — use this OR 4A, not both |
| 5 | Cell 5 | Download your generated video (only needed if you used 4A) |

Before starting: **Runtime -> Change runtime type -> T4 GPU**.

---
## STEP 1 — Cell 1: Install condacolab
This will automatically restart the runtime when done. That's expected — just wait for it to reconnect, then continue to Cell 2. **Do not re-run this cell after the restart.**

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()
# Runtime restarts automatically here. Once reconnected, move on to Cell 2.

---
## STEP 2 — Cell 2: Create Python 3.10 env, install SadTalker + all fixes, download models
Run this only AFTER the automatic restart from Cell 1. Takes ~5-8 minutes.

In [ ]:
import condacolab
condacolab.check()  # confirms conda is ready after the restart

# --- Create isolated Python 3.10 environment ---
!mamba create -n sadtalker python=3.10 -y -q

# --- Install matching torch build (CUDA 11.8, works on T4) ---
!source activate sadtalker && pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

# --- Clone SadTalker ---
!git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker
%cd /content/SadTalker

# --- Install SadTalker's own requirements ---
!source activate sadtalker && pip install -q -r requirements.txt

# --- PATCH 1: fix 'No module named pkg_resources' (newer setuptools removed it) ---
!source activate sadtalker && pip install -q "setuptools<81"

# --- PATCH 2: use Colab's system ffmpeg instead of conda-forge's (which has a broken libx264 link) ---
!source activate sadtalker && conda remove -y -q ffmpeg --force 2>/dev/null || true
!ln -sf /usr/bin/ffmpeg /usr/local/envs/sadtalker/bin/ffmpeg
!ln -sf /usr/bin/ffprobe /usr/local/envs/sadtalker/bin/ffprobe
!/usr/local/envs/sadtalker/bin/ffmpeg -version | head -n 1

# --- Download pretrained model checkpoints (official script) ---
!bash scripts/download_models.sh

print("\nSetup complete. Continue to Cell 3.")

---
## STEP 3 — Cell 3: Upload your avatar image and audio file
(Skip this if you plan to only use the Gradio web UI in Cell 4B — the UI has its own upload widgets.)

In [ ]:
from google.colab import files
import os

os.makedirs('/content/inputs', exist_ok=True)

print("Upload your AVATAR IMAGE (jpg/png):")
img = files.upload()
img_name = list(img.keys())[0]
img_path = '/content/inputs/' + img_name
os.rename(img_name, img_path)

print("\nUpload your AUDIO FILE (wav/mp3):")
aud = files.upload()
aud_name = list(aud.keys())[0]
aud_path = '/content/inputs/' + aud_name
os.rename(aud_name, aud_path)

print(f"\nImage saved to: {img_path}")
print(f"Audio saved to: {aud_path}")

---
## STEP 4A — Command-line inference (Option A)
Use this if you just want a quick one-off render with fixed settings. Skip to 4B instead if you want the web UI.

**Flags explained:**
- `--still` — minimal head motion (cleaner for simple avatar talking, fewer artifacts)
- `--preprocess full` — uses the full image, not just a cropped face (better for avatar-style shots)
- `--enhancer gfpgan` — sharpens the face (uses more VRAM/time — remove this first if you hit an out-of-memory error)
- `--size 512` — full 512px output (heavier than default 256px — remove if you hit OOM)

In [ ]:
%cd /content/SadTalker

# MPLBACKEND=Agg fixes a matplotlib backend crash caused by Colab's notebook display settings leaking into the script
!source activate sadtalker && MPLBACKEND=Agg python inference.py \
  --driven_audio /content/inputs/{aud_path.split('/')[-1]} \
  --source_image /content/inputs/{img_path.split('/')[-1]} \
  --result_dir /content/results \
  --still \
  --preprocess full \
  --enhancer gfpgan \
  --size 512

---
## STEP 4B — Gradio Web UI (Option B — easier, recommended)
Drag-and-drop image + audio upload, dropdown settings, Generate button, and video preview — all in a browser tab. No code editing needed per run.

Running this cell starts a live server — it will keep running (that's normal). A public `*.gradio.live` link will be printed below; click it to open the UI. Stop the cell when you're done to shut the server down.

In [ ]:
%cd /content/SadTalker

!source activate sadtalker && pip install -q gradio
!source activate sadtalker && MPLBACKEND=Agg python app_sadtalker.py --share

---
## STEP 5 — Download your generated video
Only needed if you used Option A (4A). If you used the Gradio UI (4B), download directly from the browser interface instead.

In [ ]:
import glob
import os
from google.colab import files

output_files = glob.glob('/content/results/**/*.mp4', recursive=True)
output_files.sort(key=lambda x: -os.path.getmtime(x))

if output_files:
    latest = output_files[0]
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No output video found — check Cell 4A for errors.")

---
### Troubleshooting reference
| Symptom | Cause | Fix |
|---|---|---|
| `condacolab.install()` disconnects | Expected — forces a restart | Wait, reconnect, continue from Cell 2 |
| `ModuleNotFoundError: pkg_resources` | Newer setuptools removed it | Already patched in Cell 2 (`setuptools<81`) |
| `ValueError: Key backend...` (matplotlib) | Colab's display backend leaks into the script | Already patched — `MPLBACKEND=Agg` prefix on every python run |
| `ffmpeg: error while loading shared libraries: libx264...` | conda-forge ffmpeg build mismatch | Already patched — Cell 2 symlinks Colab's system ffmpeg instead |
| `OpenCV: FFMPEG: tag ... is not supported ... fallback to use tag 'mp4v'` | Harmless codec fallback warning | Ignore — video still generates correctly |
| CUDA out of memory | Resolution/enhancer too heavy for T4's ~16GB | Remove `--enhancer gfpgan` first, then try `--size 256` instead of 512 |
| `download_models.sh` fails / 404s | Model checkpoint URLs moved | Check `github.com/OpenTalker/SadTalker` releases page, update URLs, or download manually into `/content/SadTalker/checkpoints` |
| Session disconnects mid-batch | Free Colab usage limits | Process clips in smaller batches per session, or upgrade to Colab Pro |